In [0]:
# Databricks notebook source
"""
Remote usability topic modeling pipeline for Buzzmetrix.

Use case
--------
- Start with one product/category slice such as "리모컨 사용성"
- Split by sc_measurement: 1 / 0 / -1
- Sample by sentiment group:
  - >=1000 reviews: deterministic sample 1000
  - 100~999 reviews: use all
  - <100 reviews: mark as not classifiable
- Build fixed topic categories for sampled reviews only
- Label each sampled review
- Enforce minimum category size and maximum representative topic count

Model recommendation
--------------------
- Taxonomy design: GPT-5.4
- Per-review labeling: GPT-5.4-mini

Note
----
This pipeline is structured for future expansion:
- common few-shot principles for labeling style
- category-specific topic guide for domain hints
"""

from __future__ import annotations

import json
import re
import time
from typing import Any, Dict, List, Optional, Tuple

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


# =========================
# Config
# =========================
PROMPT_VERSION = "remote_usability_v1_0_0"
RUN_SCOPE = "TEST"

STAGE1_ENDPOINT = "databricks-gpt-5-4"
STAGE2_ENDPOINT = "databricks-gpt-5-4-mini"

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"
OUTPUT_PREFIX = f"sandbox.z_jungryo_lee.remote_usability_topic_{PROMPT_VERSION}"

GROUP_STATUS_TABLE = f"{OUTPUT_PREFIX}_group_status"
SAMPLED_REVIEW_TABLE = f"{OUTPUT_PREFIX}_sampled_reviews"
TOPIC_DRAFT_TABLE = f"{OUTPUT_PREFIX}_topic_draft"
LABELED_SAMPLE_TABLE = f"{OUTPUT_PREFIX}_labeled_sample"
SUMMARY_TABLE = f"{OUTPUT_PREFIX}_summary"
FAILED_TABLE = f"{OUTPUT_PREFIX}_failed"

SEED = "seed_20260416"
MAX_RETRIES = 4
BACKOFF_BASE = 1.8
RATE_LIMIT_SECONDS = 0.6
MAX_TOKENS_STAGE1 = 2200
MAX_TOKENS_STAGE2 = 1400

DEFAULT_CATE_1_DEPTH = "07. 스마트 사용성"
DEFAULT_CATE_2_DEPTH = "07-06. 리모컨 사용성"

# Common few-shot examples for style and decision rules.
# These are intentionally category-agnostic and should be reused across domains.
COMMON_FEW_SHOT_EXAMPLES: List[Dict[str, Any]] = [
    {
        "principle": "simple_positive",
        "input_style": "좋아요 / 최고예요 / 만족해요처럼 이유 없는 짧은 칭찬",
        "output_style": {
            "main_topic": "전반적긍정",
            "sub_topic": "전반 평가가 좋음",
            "is_simple_eval": "Y",
        },
    },
    {
        "principle": "simple_negative",
        "input_style": "별로예요 / 구려요 / 싫어요처럼 이유 없는 짧은 혹평",
        "output_style": {
            "main_topic": "전반적부정",
            "sub_topic": "전반 평가가 나쁨",
            "is_simple_eval": "Y",
        },
    },
    {
        "principle": "specific_issue",
        "input_style": "기능, 서비스, 사용상황, 원인, 비교가 드러나는 문장",
        "output_style": {
            "main_topic": "실제 기능/서비스 주제",
            "sub_topic": "~~이 ~~함",
            "is_simple_eval": "N",
        },
    },
    {
        "principle": "other_bucket",
        "input_style": "표본 수가 너무 적거나 하나의 주제로 묶기 어려운 잔여 이슈",
        "output_style": {
            "main_topic": "기타(aaa, bbb, ccc)",
            "sub_topic": "기타(aaa, bbb, ccc)",
            "is_simple_eval": "N",
        },
    },
]

REMOTE_USABILITY_TOPIC_GUIDE: List[Dict[str, Any]] = [
    {"main_topic": "레이아웃/키", "sub_topic_hint": "버튼 배치가 복잡함"},
    {"main_topic": "레이아웃/키", "sub_topic_hint": "핵심 버튼이 없어 불편함"},
    {"main_topic": "레이아웃/키", "sub_topic_hint": "숫자키가 없어 불편함"},
    {"main_topic": "레이아웃/키", "sub_topic_hint": "미니멀 정책이 비호감임"},
    {"main_topic": "포인터", "sub_topic_hint": "포인터 조작이 비호감임"},
    {"main_topic": "포인터", "sub_topic_hint": "포인터 설정 제어가 불편함"},
    {"main_topic": "디자인", "sub_topic_hint": "품질감이 낮아 보임"},
    {"main_topic": "디자인", "sub_topic_hint": "버튼 가독성이 낮음"},
    {"main_topic": "디자인", "sub_topic_hint": "인체공학 구성이 불편함"},
    {"main_topic": "매직 장입", "sub_topic_hint": "매직리모컨이 기본 제공되지 않음"},
    {"main_topic": "매직 장입", "sub_topic_hint": "연동 기능이 지원되지 않음"},
    {"main_topic": "반응성", "sub_topic_hint": "입력 반응이 지연됨"},
    {"main_topic": "접근성", "sub_topic_hint": "고령자 사용이 어려움"},
    {"main_topic": "전반적부정", "sub_topic_hint": "전반 평가가 나쁨"},
]

print(
    f"[CONFIG] prompt_version={PROMPT_VERSION}, run_scope={RUN_SCOPE}, "
    f"stage1={STAGE1_ENDPOINT}, stage2={STAGE2_ENDPOINT}"
)


# =========================
# Utilities
# =========================
def clean_text(x: Any) -> str:
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def sc_to_sentiment(sc_value: Any) -> str:
    try:
        value = int(sc_value)
    except Exception:
        return "기타"
    if value == 1:
        return "긍정"
    if value == 0:
        return "중립"
    if value == -1:
        return "부정"
    return "기타"


def table_name_for_mode(base_name: str, test_mode: bool) -> str:
    return f"{base_name}_test" if test_mode else base_name


def safe_list(value: Any) -> List[Any]:
    return value if isinstance(value, list) else []


def normalize_json_candidate(text: str) -> str:
    candidate = clean_text(text)
    candidate = candidate.replace("\u201c", '"').replace("\u201d", '"')
    candidate = candidate.replace("\u2018", "'").replace("\u2019", "'")
    candidate = re.sub(r",\s*}", "}", candidate)
    candidate = re.sub(r",\s*]", "]", candidate)
    return candidate


def find_balanced_json_object(text: str) -> str:
    start = text.find("{")
    if start < 0:
        raise ValueError("No opening brace found")

    in_string = False
    escaped = False
    depth = 0
    for idx in range(start, len(text)):
        ch = text[idx]
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:idx + 1]
    raise ValueError("No balanced JSON object found")


def extract_json_object(text: str) -> Dict[str, Any]:
    raw = text if text is not None else ""
    candidates: List[str] = []
    stripped = raw.strip()
    if stripped:
        candidates.append(stripped)

    fenced = re.search(r"```json\s*(\{.*?\})\s*```", raw, flags=re.S)
    if fenced:
        candidates.append(fenced.group(1))

    try:
        candidates.append(find_balanced_json_object(raw))
    except Exception:
        pass

    greedy = re.search(r"\{.*\}", raw, flags=re.S)
    if greedy:
        candidates.append(greedy.group(0))

    last_error: Optional[Exception] = None
    seen = set()
    for candidate in candidates:
        normalized = normalize_json_candidate(candidate)
        if not normalized or normalized in seen:
            continue
        seen.add(normalized)
        try:
            return json.loads(normalized)
        except Exception as exc:
            last_error = exc

    raise ValueError(f"Invalid JSON from model: {repr(last_error)} | raw={clean_text(raw)[:1000]}")


def extract_response_content(resp: Any) -> str:
    if isinstance(resp, dict):
        if "choices" in resp and resp["choices"]:
            return clean_text(resp["choices"][0]["message"]["content"])
        if "predictions" in resp and resp["predictions"]:
            pred0 = resp["predictions"][0]
            if isinstance(pred0, dict) and "content" in pred0:
                return clean_text(pred0["content"])
            if isinstance(pred0, str):
                return clean_text(pred0)
        if "content" in resp:
            return clean_text(resp["content"])
    if isinstance(resp, str):
        return clean_text(resp)
    raise ValueError(f"Unexpected response schema: {type(resp)}")


def call_endpoint(endpoint: str, messages: List[Dict[str, str]], max_tokens: int) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client

    client = get_deploy_client("databricks")
    payload = {
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": max_tokens,
    }

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.predict(endpoint=endpoint, inputs=payload)
            return extract_json_object(extract_response_content(resp))
        except Exception as exc:
            last_error = exc
            try:
                repair_payload = {
                    "messages": messages + [
                        {"role": "assistant", "content": "이전 응답이 JSON 형식에 맞지 않았습니다."},
                        {
                            "role": "user",
                            "content": "설명 없이 JSON 객체 하나만 다시 출력하세요. 마크다운과 코드블록 없이 순수 JSON만 반환하세요.",
                        },
                    ],
                    "temperature": 0.0,
                    "max_tokens": max_tokens,
                }
                repair_resp = client.predict(endpoint=endpoint, inputs=repair_payload)
                return extract_json_object(extract_response_content(repair_resp))
            except Exception as repair_exc:
                last_error = repair_exc
            print(f"[LLM ERROR] endpoint={endpoint}, attempt={attempt + 1}/{MAX_RETRIES}, error={repr(last_error)}")
            time.sleep(BACKOFF_BASE ** attempt)

    raise RuntimeError(f"Endpoint request failed after retries: {last_error}")


def save_table(df: DataFrame, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] overwrite -> {table_name}")
    else:
        print(f"[SAVE] create -> {table_name}")
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def empty_failed_df() -> DataFrame:
    schema = """
        cate_1_depth string,
        cate_2_depth string,
        sc_measurement int,
        sentiment string,
        stage string,
        endpoint string,
        review_id string,
        error_message string,
        payload_key string,
        memo string
    """
    return spark.createDataFrame([], schema)


# =========================
# Data load and sampling
# =========================
def build_group_status_sql(cate_1_depth: str, cate_2_depth: str) -> str:
    return f"""
    with base as (
        select distinct
            cate_1_depth,
            cate_2_depth,
            cast(sc_measurement as int) as sc_measurement,
            memo
        from {SOURCE_TABLE}
        where 1=1
          and cate_1_depth = '{cate_1_depth}'
          and cate_2_depth = '{cate_2_depth}'
          and cate_1_depth not like '***%%'
          and memo is not null
          and length(trim(memo)) > 0
          and cast(sc_measurement as int) in (-1, 0, 1)
    )
    select
        cate_1_depth,
        cate_2_depth,
        sc_measurement,
        count(*) as review_cnt,
        case
            when count(*) >= 1000 then 1000
            when count(*) >= 100 then count(*)
            else 0
        end as sample_cnt,
        case
            when count(*) >= 1000 then 'sample_1000'
            when count(*) >= 100 then 'use_all'
            else '리뷰수 100개 미만으로 주제분류 불가'
        end as sample_rule,
        case
            when count(*) >= 1000 then 10
            when count(*) >= 100 then 5
            else null
        end as min_topic_size,
        case
            when count(*) >= 1000 then 17
            when count(*) >= 100 then 12
            else 0
        end as max_topic_cnt
    from base
    group by cate_1_depth, cate_2_depth, sc_measurement
    order by sc_measurement desc
    """


def build_sample_sql(cate_1_depth: str, cate_2_depth: str) -> str:
    return f"""
    with base as (
        select distinct
            cate_1_depth,
            cate_2_depth,
            cast(sc_measurement as int) as sc_measurement,
            memo
        from {SOURCE_TABLE}
        where 1=1
          and cate_1_depth = '{cate_1_depth}'
          and cate_2_depth = '{cate_2_depth}'
          and cate_1_depth not like '***%%'
          and memo is not null
          and length(trim(memo)) > 0
          and cast(sc_measurement as int) in (-1, 0, 1)
    ),
    grouped as (
        select
            *,
            count(*) over (
                partition by cate_1_depth, cate_2_depth, sc_measurement
            ) as review_cnt
        from base
    ),
    sampled as (
        select
            *,
            row_number() over (
                partition by cate_1_depth, cate_2_depth, sc_measurement
                order by md5(
                    concat(
                        coalesce(cate_1_depth, ''), '||',
                        coalesce(cate_2_depth, ''), '||',
                        coalesce(sc_measurement, ''), '||',
                        coalesce(memo, ''), '||',
                        '{SEED}'
                    )
                )
            ) as rn
        from grouped
    )
    select
        sha2(concat_ws('||', cate_1_depth, cate_2_depth, cast(sc_measurement as string), memo), 256) as review_id,
        cate_1_depth,
        cate_2_depth,
        sc_measurement,
        case
            when sc_measurement = 1 then '긍정'
            when sc_measurement = 0 then '중립'
            when sc_measurement = -1 then '부정'
            else '기타'
        end as sentiment,
        memo,
        review_cnt,
        case
            when review_cnt >= 1000 then 1000
            when review_cnt >= 100 then review_cnt
            else 0
        end as sample_cnt,
        case
            when review_cnt >= 1000 then 10
            when review_cnt >= 100 then 5
            else null
        end as min_topic_size,
        case
            when review_cnt >= 1000 then 17
            when review_cnt >= 100 then 12
            else 0
        end as max_topic_cnt,
        case
            when review_cnt >= 1000 and rn <= 1000 then 1
            when review_cnt between 100 and 999 then 1
            else 0
        end as use_flag
    from sampled
    where review_cnt >= 100
      and (
        (review_cnt >= 1000 and rn <= 1000)
        or (review_cnt between 100 and 999)
      )
    """


def load_group_status_df(cate_1_depth: str, cate_2_depth: str) -> DataFrame:
    return spark.sql(build_group_status_sql(cate_1_depth, cate_2_depth)).withColumn(
        "sentiment", F.udf(sc_to_sentiment, "string")(F.col("sc_measurement"))
    )


def load_sampled_reviews_df(cate_1_depth: str, cate_2_depth: str) -> DataFrame:
    return spark.sql(build_sample_sql(cate_1_depth, cate_2_depth))


# =========================
# Prompt builders
# =========================
def build_design_messages(
    row: Dict[str, Any],
    sample_memos: List[str],
    common_few_shot_examples: List[Dict[str, Any]],
) -> List[Dict[str, str]]:
    if row["sentiment"] == "긍정":
        general_topic_rule = "아주 짧은 단순 칭찬은 전반적긍정으로 분류한다."
    elif row["sentiment"] == "부정":
        general_topic_rule = "아주 짧은 단순 불만은 전반적부정으로 분류한다."
    else:
        general_topic_rule = "중립 리뷰는 전반적긍정/전반적부정 대신 실제 내용 중심으로 분류한다."

    system = f"""
당신은 한국어 VOC taxonomy 설계 전문가다.
리뷰가 다국어이면 먼저 한국어로 이해한 뒤 작업하라.

현재 고정 그룹:
- cate_1_depth = {row['cate_1_depth']}
- cate_2_depth = {row['cate_2_depth']}
- sentiment = {row['sentiment']}
- sample_cnt = {row['sample_cnt']}
- min_topic_size = {row['min_topic_size']}
- max_topic_cnt = {row['max_topic_cnt']}

목표:
1. 샘플 리뷰를 묶는 대표 주제 pool을 설계한다.
2. 대표 주제 개수는 특수 카테고리 포함 최대 {row['max_topic_cnt']}개를 넘지 않는다.
3. 각 대표 주제는 실제 리뷰가 최소 {row['min_topic_size']}개 이상 포함될 수 있게 충분히 넓고 명확해야 한다.
4. 짧은 단순 평가만 전반적긍정 또는 전반적부정으로 둔다.
5. 묶이지 않는 잔여 주제는 하나의 기타(aaa, bbb, ccc) 형식으로 설계한다.
6. 대주제는 명사형으로 쓴다.
7. 소주제는 짧은 문장형이며 "~~이 ~~함" 스타일로 쓴다.
8. 몇몇 소주제가 큰 영역으로 자연스럽게 묶이면 그때만 대주제로 묶고, 아니면 대주제와 소주제가 같아도 된다.
9. JSON 객체 하나만 반환한다.

추가 규칙:
- {general_topic_rule}
- 좋아요/최고예요/별로예요/구려요 같이 이유 없는 초단문은 전반 범주로 둔다.
- 리모컨 사용성에서는 아래 주제 가이드를 우선 참고하되, 실제 샘플에 없는 주제는 억지로 만들지 않는다.
- 주제 가이드:
  - 레이아웃/키: 복잡한 조작, 버튼 배치 난해, 학습 어려움, 핵심 버튼 부재, 숫자키 부재, 미니멀/AI 밀어붙임 비호감
  - 포인터: 포인터 방식 비호감, 커서 민감도/자동표시/비활성 제어 불만
  - 디자인: 저가/구형/품질감 낮음, 버튼 가독성/백라이트/크기 문제, 인체공학/레이아웃 비호감
  - 매직 장입: 매직리모컨 미동봉, 별도 구매 불만, 호환/연동/음성/알렉사 미지원
  - 반응성: 지연, 렉, 과민 반응, 하드웨어 불량
  - 접근성: 고령자, 손떨림, 시각 접근성 문제
  - 전반적부정: 짧은 혹평 문구
- 너무 세분화된 주제는 금지한다.
- 기타는 반드시 한 개만 만든다.

출력 스키마:
{{
  "topics": [
    {{
      "main_topic": "대주제",
      "sub_topic": "소주제",
      "simple_eval_only": "Y|N"
    }}
  ],
  "other_bucket_label": "기타(aaa, bbb, ccc)"
}}
"""
    user = (
        f"common_few_shot_examples:\n{json.dumps(common_few_shot_examples, ensure_ascii=False)}\n\n"
        f"remote_usability_topic_guide:\n{json.dumps(REMOTE_USABILITY_TOPIC_GUIDE, ensure_ascii=False)}\n\n"
        f"sample_reviews:\n{json.dumps(sample_memos, ensure_ascii=False)}"
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_label_messages(
    row: Dict[str, Any],
    topic_pool: Dict[str, Any],
    common_few_shot_examples: List[Dict[str, Any]],
) -> List[Dict[str, str]]:
    system = f"""
당신은 한국어 VOC 분류기다.
리뷰가 다국어이면 먼저 한국어로 이해한 뒤 분류하라.

현재 고정 그룹:
- cate_1_depth = {row['cate_1_depth']}
- cate_2_depth = {row['cate_2_depth']}
- sentiment = {row['sentiment']}

반드시 아래 규칙을 지켜라.
1. 주어진 주제 pool 안에서만 분류한다.
2. 아주 짧고 이유 없는 단순 칭찬/불만만 전반적긍정 또는 전반적부정으로 분류한다.
3. 기능, 사용 상황, 버튼, 반응, 인식, 디자인, 편의성 등 구체 내용이 있으면 실제 주제로 분류한다.
4. 출력은 JSON 객체 하나만 반환한다.
5. main_topic은 명사형이어야 한다.
6. sub_topic은 "~~이 ~~함" 스타일 또는 주어진 기타 라벨 그대로 사용한다.

출력 스키마:
{{
  "memo_ko": "한국어 해석",
  "main_topic": "대주제",
  "sub_topic": "소주제",
  "is_simple_eval": "Y|N",
  "evidence": "짧은 근거"
}}
"""
    user = (
        f"common_few_shot_examples:\n{json.dumps(common_few_shot_examples, ensure_ascii=False)}\n\n"
        f"remote_usability_topic_guide:\n{json.dumps(REMOTE_USABILITY_TOPIC_GUIDE, ensure_ascii=False)}\n\n"
        f"topic_pool:\n{json.dumps(topic_pool, ensure_ascii=False)}\n\n"
        f"review:\n{row['memo']}"
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


# =========================
# Stage 1: design per sentiment group
# =========================
def design_topic_pool(
    sampled_df: DataFrame,
    group_status_df: DataFrame,
    common_few_shot_examples: List[Dict[str, Any]],
) -> Tuple[DataFrame, DataFrame]:
    sampled_rows = [r.asDict() for r in sampled_df.orderBy("sc_measurement").toLocalIterator()]
    status_rows = {
        int(r["sc_measurement"]): r.asDict()
        for r in group_status_df.where(F.col("sample_cnt") > 0).toLocalIterator()
    }

    by_sc: Dict[int, List[Dict[str, Any]]] = {}
    for row in sampled_rows:
        by_sc.setdefault(int(row["sc_measurement"]), []).append(row)

    draft_rows: List[Dict[str, Any]] = []
    failed_rows: List[Dict[str, Any]] = []

    for sc_measurement, rows in by_sc.items():
        status = status_rows[sc_measurement]
        memos = [clean_text(x["memo"]) for x in rows]
        try:
            result = call_endpoint(
                endpoint=STAGE1_ENDPOINT,
                messages=build_design_messages(status, memos, common_few_shot_examples),
                max_tokens=MAX_TOKENS_STAGE1,
            )
            other_bucket_label = clean_text(result.get("other_bucket_label")) or "기타(기타 이슈)"
            for order_idx, topic in enumerate(safe_list(result.get("topics")), start=1):
                main_topic = clean_text(topic.get("main_topic"))
                sub_topic = clean_text(topic.get("sub_topic"))
                if not main_topic or not sub_topic:
                    continue
                draft_rows.append(
                    {
                        "cate_1_depth": status["cate_1_depth"],
                        "cate_2_depth": status["cate_2_depth"],
                        "sc_measurement": sc_measurement,
                        "sentiment": status["sentiment"],
                        "review_cnt": int(status["review_cnt"]),
                        "sample_cnt": int(status["sample_cnt"]),
                        "min_topic_size": int(status["min_topic_size"]),
                        "max_topic_cnt": int(status["max_topic_cnt"]),
                        "topic_order": order_idx,
                        "main_topic": main_topic,
                        "sub_topic": sub_topic,
                        "simple_eval_only": clean_text(topic.get("simple_eval_only")) or "N",
                        "other_bucket_label": other_bucket_label,
                    }
                )
        except Exception as exc:
            failed_rows.append(
                {
                    "cate_1_depth": status["cate_1_depth"],
                    "cate_2_depth": status["cate_2_depth"],
                    "sc_measurement": sc_measurement,
                    "sentiment": status["sentiment"],
                    "stage": "design_topic_pool",
                    "endpoint": STAGE1_ENDPOINT,
                    "review_id": "",
                    "error_message": clean_text(exc),
                    "payload_key": f"{status['cate_1_depth']}||{status['cate_2_depth']}||{sc_measurement}",
                    "memo": "",
                }
            )
        time.sleep(RATE_LIMIT_SECONDS)

    draft_schema = """
        cate_1_depth string,
        cate_2_depth string,
        sc_measurement int,
        sentiment string,
        review_cnt int,
        sample_cnt int,
        min_topic_size int,
        max_topic_cnt int,
        topic_order int,
        main_topic string,
        sub_topic string,
        simple_eval_only string,
        other_bucket_label string
    """
    draft_df = spark.createDataFrame(draft_rows, draft_schema) if draft_rows else spark.createDataFrame([], draft_schema)
    failed_df = spark.createDataFrame(failed_rows, empty_failed_df().schema) if failed_rows else empty_failed_df()
    return draft_df, failed_df


def build_topic_pool_lookup(topic_draft_df: DataFrame) -> Dict[int, Dict[str, Any]]:
    rows = [r.asDict() for r in topic_draft_df.orderBy("sc_measurement", "topic_order").toLocalIterator()]
    lookup: Dict[int, Dict[str, Any]] = {}
    for row in rows:
        sc_value = int(row["sc_measurement"])
        lookup.setdefault(
            sc_value,
            {
                "topics": [],
                "other_bucket_label": row["other_bucket_label"],
                "min_topic_size": int(row["min_topic_size"]),
                "max_topic_cnt": int(row["max_topic_cnt"]),
            },
        )
        lookup[sc_value]["topics"].append(
            {
                "main_topic": row["main_topic"],
                "sub_topic": row["sub_topic"],
                "simple_eval_only": row["simple_eval_only"],
            }
        )
    return lookup


# =========================
# Stage 2: label sampled reviews
# =========================
def label_sampled_reviews(
    sampled_df: DataFrame,
    topic_lookup: Dict[int, Dict[str, Any]],
    common_few_shot_examples: List[Dict[str, Any]],
) -> Tuple[DataFrame, DataFrame]:
    rows = [r.asDict() for r in sampled_df.orderBy("sc_measurement").toLocalIterator()]

    labeled_rows: List[Dict[str, Any]] = []
    failed_rows: List[Dict[str, Any]] = []

    for idx, row in enumerate(rows, start=1):
        sc_value = int(row["sc_measurement"])
        topic_pool = topic_lookup.get(sc_value)
        if not topic_pool:
            failed_rows.append(
                {
                    "cate_1_depth": row["cate_1_depth"],
                    "cate_2_depth": row["cate_2_depth"],
                    "sc_measurement": sc_value,
                    "sentiment": row["sentiment"],
                    "stage": "label_sampled_reviews_missing_pool",
                    "endpoint": STAGE2_ENDPOINT,
                    "review_id": row["review_id"],
                    "error_message": "topic_pool_not_found",
                    "payload_key": f"{row['cate_1_depth']}||{row['cate_2_depth']}||{sc_value}",
                    "memo": row["memo"],
                }
            )
            continue

        try:
            result = call_endpoint(
                endpoint=STAGE2_ENDPOINT,
                messages=build_label_messages(row, topic_pool, common_few_shot_examples),
                max_tokens=MAX_TOKENS_STAGE2,
            )
            labeled_rows.append(
                {
                    "review_id": row["review_id"],
                    "cate_1_depth": row["cate_1_depth"],
                    "cate_2_depth": row["cate_2_depth"],
                    "sc_measurement": sc_value,
                    "sentiment": row["sentiment"],
                    "review_cnt": int(row["review_cnt"]),
                    "sample_cnt": int(row["sample_cnt"]),
                    "min_topic_size": int(row["min_topic_size"]),
                    "max_topic_cnt": int(row["max_topic_cnt"]),
                    "memo": row["memo"],
                    "memo_ko": clean_text(result.get("memo_ko")),
                    "main_topic": clean_text(result.get("main_topic")) or "기타",
                    "sub_topic": clean_text(result.get("sub_topic")) or topic_pool.get("other_bucket_label", "기타(기타 이슈)"),
                    "is_simple_eval": clean_text(result.get("is_simple_eval")) or "N",
                    "evidence": clean_text(result.get("evidence")),
                }
            )
        except Exception as exc:
            failed_rows.append(
                {
                    "cate_1_depth": row["cate_1_depth"],
                    "cate_2_depth": row["cate_2_depth"],
                    "sc_measurement": sc_value,
                    "sentiment": row["sentiment"],
                    "stage": "label_sampled_reviews",
                    "endpoint": STAGE2_ENDPOINT,
                    "review_id": row["review_id"],
                    "error_message": clean_text(exc),
                    "payload_key": f"{row['cate_1_depth']}||{row['cate_2_depth']}||{sc_value}",
                    "memo": row["memo"],
                }
            )

        if idx % 100 == 0:
            print(f"[LABEL] processed={idx}")
            time.sleep(RATE_LIMIT_SECONDS)

    schema = """
        review_id string,
        cate_1_depth string,
        cate_2_depth string,
        sc_measurement int,
        sentiment string,
        review_cnt int,
        sample_cnt int,
        min_topic_size int,
        max_topic_cnt int,
        memo string,
        memo_ko string,
        main_topic string,
        sub_topic string,
        is_simple_eval string,
        evidence string
    """
    labeled_df = spark.createDataFrame(labeled_rows, schema) if labeled_rows else spark.createDataFrame([], schema)
    failed_df = spark.createDataFrame(failed_rows, empty_failed_df().schema) if failed_rows else empty_failed_df()
    return labeled_df, failed_df


# =========================
# Post-processing rules
# =========================
def enforce_topic_rules(
    labeled_df: DataFrame,
    topic_draft_df: DataFrame,
) -> DataFrame:
    other_label_df = (
        topic_draft_df.select("sc_measurement", "other_bucket_label")
        .dropDuplicates(["sc_measurement"])
    )
    labeled_df = labeled_df.join(other_label_df, ["sc_measurement"], "left")
    labeled_df.createOrReplaceTempView("tmp_labeled_before_rules")

    sql = """
    with counted as (
        select
            *,
            count(*) over (
                partition by cate_1_depth, cate_2_depth, sc_measurement, sub_topic
            ) as sub_topic_cnt
        from tmp_labeled_before_rules
    ),
    sparse_merged as (
        select
            review_id,
            cate_1_depth,
            cate_2_depth,
            sc_measurement,
            sentiment,
            review_cnt,
            sample_cnt,
            min_topic_size,
            max_topic_cnt,
            memo,
            memo_ko,
            case
                when sub_topic_cnt < min_topic_size then coalesce(other_bucket_label, '기타(기타 이슈)')
                else main_topic
            end as main_topic_tmp,
            case
                when sub_topic_cnt < min_topic_size then coalesce(other_bucket_label, '기타(기타 이슈)')
                else sub_topic
            end as sub_topic_tmp,
            is_simple_eval,
            evidence,
            coalesce(other_bucket_label, '기타(기타 이슈)') as other_bucket_label
        from counted
    ),
    ranked as (
        select
            *,
            count(*) over (
                partition by cate_1_depth, cate_2_depth, sc_measurement, sub_topic_tmp
            ) as final_sub_topic_cnt
        from sparse_merged
    ),
    topic_rank as (
        select distinct
            cate_1_depth,
            cate_2_depth,
            sc_measurement,
            sub_topic_tmp,
            final_sub_topic_cnt,
            row_number() over (
                partition by cate_1_depth, cate_2_depth, sc_measurement
                order by
                    case
                        when sub_topic_tmp like '기타(%' then 1
                        when sub_topic_tmp = '전반 평가가 좋음' then 1
                        when sub_topic_tmp = '전반 평가가 나쁨' then 1
                        else 0
                    end desc,
                    final_sub_topic_cnt desc,
                    sub_topic_tmp
            ) as topic_rank
        from ranked
    ),
    rank_with_limit as (
        select
            tr.*,
            ml.max_topic_limit
        from topic_rank tr
        left join (
            select distinct
                cate_1_depth,
                cate_2_depth,
                sc_measurement,
                max_topic_cnt as max_topic_limit
            from ranked
        ) ml
          on tr.cate_1_depth = ml.cate_1_depth
         and tr.cate_2_depth = ml.cate_2_depth
         and tr.sc_measurement = ml.sc_measurement
    ),
    keep_topics as (
        select
            cate_1_depth,
            cate_2_depth,
            sc_measurement,
            sub_topic_tmp
        from rank_with_limit
        where topic_rank <= max_topic_limit
    ),
    final as (
        select
            r.review_id,
            r.cate_1_depth,
            r.cate_2_depth,
            r.sc_measurement,
            r.sentiment,
            r.review_cnt,
            r.sample_cnt,
            r.min_topic_size,
            r.max_topic_cnt,
            r.memo,
            r.memo_ko,
            case
                when k.sub_topic_tmp is null and r.sub_topic_tmp not like '기타(%'
                    then r.other_bucket_label
                else r.sub_topic_tmp
            end as sub_topic_final,
            case
                when k.sub_topic_tmp is null and r.sub_topic_tmp not like '기타(%'
                    then r.other_bucket_label
                when r.sub_topic_tmp like '기타(%'
                    then r.sub_topic_tmp
                else r.main_topic_tmp
            end as main_topic_final,
            r.is_simple_eval,
            r.evidence
        from ranked r
        left join keep_topics k
          on r.cate_1_depth = k.cate_1_depth
         and r.cate_2_depth = k.cate_2_depth
         and r.sc_measurement = k.sc_measurement
         and r.sub_topic_tmp = k.sub_topic_tmp
    )
    select
        review_id,
        cate_1_depth,
        cate_2_depth,
        sc_measurement,
        sentiment,
        review_cnt,
        sample_cnt,
        min_topic_size,
        max_topic_cnt,
        memo,
        memo_ko,
        main_topic_final as main_topic,
        sub_topic_final as sub_topic,
        is_simple_eval,
        evidence
    from final
    """
    return spark.sql(sql)


def build_summary_df(group_status_df: DataFrame, labeled_final_df: DataFrame) -> DataFrame:
    labeled_summary = (
        labeled_final_df.groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "sentiment", "main_topic", "sub_topic")
        .agg(F.count("*").alias("memo_cnt"))
    )

    not_available = (
        group_status_df.where(F.col("sample_cnt") == 0)
        .select(
            "cate_1_depth",
            "cate_2_depth",
            "sc_measurement",
            "sentiment",
            F.lit("리뷰수 100개 미만으로 주제분류 불가").alias("main_topic"),
            F.lit("리뷰수 100개 미만으로 주제분류 불가").alias("sub_topic"),
            F.col("review_cnt").alias("memo_cnt"),
        )
    )
    return labeled_summary.unionByName(not_available, allowMissingColumns=True).orderBy(
        "sc_measurement", F.desc("memo_cnt"), "main_topic", "sub_topic"
    )


# =========================
# Orchestration
# =========================
def run_remote_usability_topic_model(
    cate_1_depth: str = DEFAULT_CATE_1_DEPTH,
    cate_2_depth: str = DEFAULT_CATE_2_DEPTH,
    common_few_shot_examples: Optional[List[Dict[str, Any]]] = None,
    test_mode: bool = True,
) -> Dict[str, DataFrame]:
    common_few_shot_examples = common_few_shot_examples or COMMON_FEW_SHOT_EXAMPLES

    group_status_df = load_group_status_df(cate_1_depth, cate_2_depth).cache()
    sampled_df = load_sampled_reviews_df(cate_1_depth, cate_2_depth).cache()

    print(f"[RUN] group_status rows = {group_status_df.count()}")
    print(f"[RUN] sampled rows = {sampled_df.count()}")

    topic_draft_df, failed_design_df = design_topic_pool(sampled_df, group_status_df, common_few_shot_examples)
    print(f"[RUN] topic_draft rows = {topic_draft_df.count()}")

    topic_lookup = build_topic_pool_lookup(topic_draft_df)
    labeled_raw_df, failed_label_df = label_sampled_reviews(sampled_df, topic_lookup, common_few_shot_examples)
    print(f"[RUN] labeled_raw rows = {labeled_raw_df.count()}")

    labeled_final_df = enforce_topic_rules(labeled_raw_df, topic_draft_df)
    summary_df = build_summary_df(group_status_df, labeled_final_df)

    failed_df = failed_design_df.unionByName(failed_label_df, allowMissingColumns=True)

    save_table(group_status_df, table_name_for_mode(GROUP_STATUS_TABLE, test_mode))
    save_table(sampled_df, table_name_for_mode(SAMPLED_REVIEW_TABLE, test_mode))
    save_table(topic_draft_df, table_name_for_mode(TOPIC_DRAFT_TABLE, test_mode))
    save_table(labeled_final_df, table_name_for_mode(LABELED_SAMPLE_TABLE, test_mode))
    save_table(summary_df, table_name_for_mode(SUMMARY_TABLE, test_mode))
    save_table(failed_df, table_name_for_mode(FAILED_TABLE, test_mode))

    print("[DISPLAY] group status")
    display(group_status_df)
    print("[DISPLAY] summary")
    display(summary_df)
    print("[DISPLAY] labeled sample")
    display(labeled_final_df.limit(100))

    group_status_df.unpersist()
    sampled_df.unpersist()

    return {
        "group_status_df": group_status_df,
        "sampled_df": sampled_df,
        "topic_draft_df": topic_draft_df,
        "labeled_final_df": labeled_final_df,
        "summary_df": summary_df,
        "failed_df": failed_df,
    }


print("[RUN] remote usability topic model test")
result = run_remote_usability_topic_model(
    cate_1_depth=DEFAULT_CATE_1_DEPTH,
    cate_2_depth=DEFAULT_CATE_2_DEPTH,
    common_few_shot_examples=COMMON_FEW_SHOT_EXAMPLES,
    test_mode=True,
)
